# Calgary Traffic Risk Intelligence

## 03 — Hotspot and Intersection Analysis

This notebook identifies recurring geographic concentrations of recorded
traffic incidents and links incidents to nearby Calgary traffic controls.

### Objectives

- Validate geographic coordinates
- Use DBSCAN to detect recurring incident concentrations
- Review DBSCAN parameter sensitivity
- Summarize hotspot characteristics
- Find the nearest traffic control to each incident
- Apply a conservative distance threshold for intersection matching
- Build traffic-control-level incident summaries
- Export the derived datasets for later weather, risk, and modelling work

> A hotspot in this project means a geographic concentration of recorded
> incidents under the selected DBSCAN parameters. It is not an official
> City of Calgary safety designation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display
from sklearn.cluster import DBSCAN
from sklearn.neighbors import BallTree

# Display tables clearly.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

## Load Cleaned Data

This notebook uses the cleaned datasets created by
`01_data_preparation.ipynb`.

In [2]:
# Find the project root.
cwd = Path.cwd()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned"

INCIDENTS_FILE = CLEAN_DIR / "incidents_clean.csv"
SIGNALS_FILE = CLEAN_DIR / "signals_clean.csv"

print("Project root:", PROJECT_ROOT)
print("Incidents file exists:", INCIDENTS_FILE.exists())
print("Signals file exists:", SIGNALS_FILE.exists())

missing_inputs = [
    path for path in [INCIDENTS_FILE, SIGNALS_FILE]
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Run 01_data_preparation.ipynb first. "
        f"Missing: {missing_inputs}"
    )

Project root: c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence
Incidents file exists: True
Signals file exists: True


In [3]:
# Load cleaned datasets.
incidents = pd.read_csv(INCIDENTS_FILE)
signals = pd.read_csv(SIGNALS_FILE)

print(f"Incidents: {len(incidents):,}")
print(f"Traffic controls: {len(signals):,}")

Incidents: 64,798
Traffic controls: 1,791


In [4]:
# Restore datetime columns from CSV.
incidents["start_dt_utc"] = pd.to_datetime(
    incidents["start_dt_utc"],
    utc=True,
)

incidents["start_dt_local"] = pd.to_datetime(
    incidents["start_dt_local"],
    utc=True,
).dt.tz_convert("America/Edmonton")

incidents["modified_dt_utc"] = pd.to_datetime(
    incidents["modified_dt_utc"],
    utc=True,
    errors="coerce",
)

signals["install_date"] = pd.to_datetime(
    signals["install_date"],
    errors="coerce",
)

print("Datetime conversion complete.")

Datetime conversion complete.


## Geographic Validation

DBSCAN and nearest-neighbour matching depend directly on latitude and
longitude, so coordinate quality is checked before spatial analysis.

In [5]:
# Check required coordinate fields.
assert incidents[["latitude", "longitude"]].notna().all().all()
assert signals[["latitude", "longitude"]].notna().all().all()

assert incidents["latitude"].between(-90, 90).all()
assert incidents["longitude"].between(-180, 180).all()

assert signals["latitude"].between(-90, 90).all()
assert signals["longitude"].between(-180, 180).all()

print("Coordinate validation passed.")

Coordinate validation passed.


In [6]:
# Review coordinate ranges.
print("Incident coordinate range")
display(
    incidents[["latitude", "longitude"]]
    .agg(["min", "max"])
)

print("Traffic-control coordinate range")
display(
    signals[["latitude", "longitude"]]
    .agg(["min", "max"])
)

Incident coordinate range


,latitude,longitude
min,50.848554,-114.309770
max,51.198505,-113.865579


Traffic-control coordinate range


,latitude,longitude
min,50.861804,-114.258956
max,51.185747,-113.888654


## DBSCAN Hotspot Detection

DBSCAN is useful here because:

- the number of hotspots does not need to be specified in advance
- clusters can have irregular shapes
- isolated incidents can be labelled as noise
- geographic density is more relevant than forcing every point into a cluster

Latitude and longitude are converted to radians and DBSCAN uses the
Haversine distance so distances are interpreted on the Earth's surface.

In [7]:
# Mean Earth radius in metres.
EARTH_RADIUS_M = 6_371_000.0

# Convert incident coordinates from degrees to radians.
incident_coords_rad = np.radians(
    incidents[["latitude", "longitude"]].to_numpy()
)

print("Coordinate matrix shape:", incident_coords_rad.shape)

Coordinate matrix shape: (64798, 2)


### Parameter Sensitivity

`eps` controls the neighbourhood radius and `min_samples` controls how
many nearby incidents are required to form dense regions.

A few nearby settings are compared before fixing the project parameters.
This makes the choice visible rather than hiding it inside the model.

In [8]:
# Compare a small set of intersection-scale DBSCAN radii.
candidate_eps_m = [40, 50, 60, 75]
MIN_SAMPLES_TEST = 20

sensitivity_rows = []

for eps_m in candidate_eps_m:
    model = DBSCAN(
        eps=eps_m / EARTH_RADIUS_M,
        min_samples=MIN_SAMPLES_TEST,
        metric="haversine",
        algorithm="ball_tree",
        n_jobs=-1,
    )

    labels = model.fit_predict(incident_coords_rad)

    cluster_count = len(set(labels)) - (1 if -1 in labels else 0)
    noise_count = int((labels == -1).sum())

    sensitivity_rows.append(
        {
            "eps_m": eps_m,
            "min_samples": MIN_SAMPLES_TEST,
            "clusters": cluster_count,
            "clustered_incidents": len(labels) - noise_count,
            "noise_incidents": noise_count,
            "noise_percent": round(noise_count / len(labels) * 100, 2),
        }
    )

dbscan_sensitivity = pd.DataFrame(sensitivity_rows)

display(dbscan_sensitivity)

,eps_m,min_samples,clusters,clustered_incidents,noise_incidents,noise_percent
0,40,20,696,35724,29074,44.87
1,50,20,696,38716,26082,40.25
2,60,20,690,41042,23756,36.66
3,75,20,668,43683,21115,32.59


### Selected Project Parameters

For this project we use:

- `eps = 50 metres`
- `min_samples = 20`

The 50 m radius is intended to capture repeated incidents at approximately
intersection/local-road scale without allowing nearby road corridors to
merge too aggressively.

The 20-incident minimum requires repeated historical activity before a
location is treated as a hotspot.

These are analytical project parameters, not official safety thresholds.

In [9]:
# Final DBSCAN parameters.
DBSCAN_EPS_M = 50
DBSCAN_MIN_SAMPLES = 20

dbscan = DBSCAN(
    eps=DBSCAN_EPS_M / EARTH_RADIUS_M,
    min_samples=DBSCAN_MIN_SAMPLES,
    metric="haversine",
    algorithm="ball_tree",
    n_jobs=-1,
)

# Assign one cluster label to every incident.
incidents["hotspot_id"] = dbscan.fit_predict(
    incident_coords_rad
)

print("DBSCAN complete.")

DBSCAN complete.


In [10]:
# Summarize cluster membership.
number_of_hotspots = (
    incidents.loc[
        incidents["hotspot_id"] != -1,
        "hotspot_id",
    ]
    .nunique()
)

noise_count = (
    incidents["hotspot_id"] == -1
).sum()

clustered_count = len(incidents) - noise_count

print("Hotspots found:", number_of_hotspots)
print("Incidents assigned to hotspots:", f"{clustered_count:,}")
print("Noise incidents:", f"{noise_count:,}")
print(
    "Clustered percentage:",
    f"{clustered_count / len(incidents) * 100:.2f}%"
)

Hotspots found: 696
Incidents assigned to hotspots: 38,716
Noise incidents: 26,082
Clustered percentage: 59.75%


## Hotspot Summaries

Each DBSCAN cluster is summarized using its incident count, geographic
centre, active years, common time patterns, and common incident description.

In [11]:
# Return the most frequent non-missing value in a Series.
def first_mode(series):
    mode = series.mode(dropna=True)

    if mode.empty:
        return pd.NA

    return mode.iloc[0]

In [12]:
# Exclude DBSCAN noise before calculating hotspot summaries.
clustered_incidents = incidents[
    incidents["hotspot_id"] != -1
].copy()

hotspot_summary = (
    clustered_incidents
    .groupby("hotspot_id")
    .agg(
        incident_count=("incident_id", "size"),
        center_latitude=("latitude", "mean"),
        center_longitude=("longitude", "mean"),
        first_incident=("start_dt_local", "min"),
        latest_incident=("start_dt_local", "max"),
        active_years=("year", "nunique"),
        most_common_weekday=("weekday", first_mode),
        most_common_hour=("hour", first_mode),
        most_common_time_period=("time_period", first_mode),
        most_common_description=("description", first_mode),
    )
    .reset_index()
    .sort_values(
        "incident_count",
        ascending=False,
    )
    .reset_index(drop=True)
)

# Rank hotspots by historical incident count.
hotspot_summary["incident_count_rank"] = (
    hotspot_summary["incident_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

display(hotspot_summary.head(20))

,hotspot_id,incident_count,center_latitude,center_longitude,first_incident,latest_incident,active_years,most_common_weekday,most_common_hour,most_common_time_period,most_common_description,incident_count_rank
0,0,1239,51.049009,-114.019265,2016-12-06 14:36:00-07:00,2026-09-25 22:22:08-06:00,11,Friday,17,Evening Rush,Two vehicle incident.,1
1,13,871,50.993939,-114.039686,2016-12-07 11:40:00-07:00,2026-09-26 16:07:22-06:00,11,Friday,17,Evening Rush,Two vehicle incident.,2
2,6,784,51.067255,-114.027703,2016-12-06 22:48:00-07:00,2026-09-13 22:58:00-06:00,11,Friday,16,Evening Rush,Two vehicle incident.,3
3,33,676,50.995895,-114.100593,2016-12-08 19:14:00-07:00,2026-09-24 14:51:34-06:00,11,Friday,15,Midday,Traffic incident.,4
4,8,427,51.110880,-114.047995,2016-12-07 08:25:00-07:00,2026-09-27 12:20:41-06:00,11,Wednesday,17,Evening Rush,Two vehicle incident.,5
5,126,390,50.999658,-114.115236,2016-12-17 14:05:00-07:00,2026-09-26 15:23:25-06:00,11,Friday,16,Evening Rush,Traffic incident.,6
6,133,382,50.998764,-114.033093,2016-12-19 15:14:00-07:00,2026-09-27 15:13:06-06:00,11,Thursday,14,Midday,Traffic incident. Blocking the right lane,7
7,94,319,51.008553,-114.019393,2016-12-14 10:18:00-07:00,2026-09-02 08:56:00-06:00,11,Friday,9,Midday,Traffic incident. Blocking the right lane,8
8,34,309,50.994296,-114.070663,2016-12-08 22:05:00-07:00,2026-09-15 17:46:00-06:00,11,Friday,17,Evening Rush,Two vehicle incident. Blocking the right lane,9
9,66,273,51.050466,-114.049933,2016-12-10 18:35:00-07:00,2026-09-16 12:56:00-06:00,11,Monday,7,Evening Rush,Traffic incident. Blocking the right lane,10


In [13]:
# Show the distribution of hotspot sizes.
display(
    hotspot_summary["incident_count"]
    .describe()
    .to_frame("incident_count")
)

,incident_count
count,696.000000
mean,55.626437
std,79.899603
min,9.000000
25%,26.000000
50%,38.000000
75%,57.000000
max,1239.000000


In [14]:
# Plot the 20 largest hotspots by recorded incident count.
top_hotspots = hotspot_summary.head(20).copy()

top_hotspots["hotspot_label"] = (
    "Hotspot "
    + top_hotspots["hotspot_id"].astype(str)
)

fig = px.bar(
    top_hotspots.sort_values("incident_count"),
    x="incident_count",
    y="hotspot_label",
    orientation="h",
    title="20 Largest DBSCAN Hotspots by Recorded Incident Count",
    labels={
        "incident_count": "Recorded incidents",
        "hotspot_label": "Hotspot",
    },
)

fig.show()

## Hotspot Map

The map displays hotspot centres rather than all 64,000+ incident points.
Marker size represents the number of incidents assigned to each hotspot.

In [15]:
# Map all detected hotspot centres.
fig = px.scatter_map(
    hotspot_summary,
    lat="center_latitude",
    lon="center_longitude",
    size="incident_count",
    hover_name="hotspot_id",
    hover_data={
        "incident_count": True,
        "most_common_weekday": True,
        "most_common_hour": True,
        "most_common_time_period": True,
        "active_years": True,
        "center_latitude": False,
        "center_longitude": False,
    },
    zoom=9,
    map_style="open-street-map",
    title="Calgary Traffic Incident Hotspots",
    size_max=25,
)

fig.show()

## Nearest Traffic-Control Matching

The next task links each incident to the nearest traffic-control location.

A nearest neighbour is always mathematically available, but that does not
mean every incident occurred at an intersection. Therefore the nearest
distance is calculated first, then a maximum distance threshold is applied.

In [16]:
# Convert traffic-control coordinates to radians.
signal_coords_rad = np.radians(
    signals[["latitude", "longitude"]].to_numpy()
)

# Build a geographic nearest-neighbour search tree.
signal_tree = BallTree(
    signal_coords_rad,
    metric="haversine",
)

# Find the nearest traffic control for every incident.
nearest_distance_rad, nearest_index = signal_tree.query(
    incident_coords_rad,
    k=1,
)

nearest_distance_m = (
    nearest_distance_rad[:, 0]
    * EARTH_RADIUS_M
)

nearest_index = nearest_index[:, 0]

print("Nearest-neighbour search complete.")

Nearest-neighbour search complete.


In [17]:
# Review nearest-control distance distribution.
distance_summary = pd.Series(
    nearest_distance_m,
    name="nearest_control_distance_m",
).describe(
    percentiles=[
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

display(distance_summary.to_frame())

,nearest_control_distance_m
count,64798.000000
mean,226.537258
std,345.159719
min,0.000882
25%,12.060720
50%,70.915825
75%,331.218323
90%,640.536097
95%,921.945833
99%,1486.489063


### Matching-Distance Sensitivity

Several thresholds are compared before choosing the final intersection
matching radius.

In [18]:
# Compare how many incidents would match at different radii.
match_thresholds_m = [30, 50, 75, 100, 125, 150]

match_sensitivity = pd.DataFrame(
    {
        "max_distance_m": match_thresholds_m,
        "matched_incidents": [
            int((nearest_distance_m <= distance).sum())
            for distance in match_thresholds_m
        ],
    }
)

match_sensitivity["matched_percent"] = (
    match_sensitivity["matched_incidents"]
    / len(incidents)
    * 100
).round(2)

display(match_sensitivity)

,max_distance_m,matched_incidents,matched_percent
0,30,21738,33.55
1,50,29330,45.26
2,75,33081,51.05
3,100,35679,55.06
4,125,37903,58.49
5,150,39377,60.77


### Selected Matching Threshold

This project uses a **75 metre** maximum distance.

An incident farther than 75 m from its nearest traffic-control point is
left unmatched rather than forcing it onto an unrelated intersection.

The threshold is a project assumption and will be documented as a
limitation.

In [19]:
# Final maximum distance for traffic-control matching.
MATCH_RADIUS_M = 75

# Store nearest-control information for every incident.
nearest_controls = (
    signals
    .iloc[nearest_index]
    .reset_index(drop=True)
)

incidents["nearest_signal_id"] = (
    nearest_controls["signal_id"].to_numpy()
)

incidents["nearest_intersection_name"] = (
    nearest_controls["intersection_name"].to_numpy()
)

incidents["nearest_control_type"] = (
    nearest_controls["control_type"].to_numpy()
)

incidents["nearest_signal_distance_m"] = (
    nearest_distance_m
)

# Mark only nearby controls as valid matches.
incidents["matched_to_signal"] = (
    incidents["nearest_signal_distance_m"]
    <= MATCH_RADIUS_M
)

print(
    "Matched incidents:",
    f"{incidents['matched_to_signal'].sum():,}"
)

print(
    "Matched percentage:",
    f"{incidents['matched_to_signal'].mean() * 100:.2f}%"
)

Matched incidents: 33,081
Matched percentage: 51.05%


In [20]:
# Create explicit matched fields while preserving nearest-control data.
incidents["matched_signal_id"] = (
    incidents["nearest_signal_id"]
    .where(incidents["matched_to_signal"])
    .astype("Int64")
)

incidents["matched_intersection_name"] = (
    incidents["nearest_intersection_name"]
    .where(incidents["matched_to_signal"])
)

incidents["matched_control_type"] = (
    incidents["nearest_control_type"]
    .where(incidents["matched_to_signal"])
)

display(
    incidents[
        [
            "incident_id",
            "incident_location",
            "hotspot_id",
            "nearest_intersection_name",
            "nearest_signal_distance_m",
            "matched_to_signal",
        ]
    ].head(10)
)

,incident_id,incident_location,hotspot_id,nearest_intersection_name,nearest_signal_distance_m,matched_to_signal
0,2016-12-06 17:00:00+00:00-114.064987451.09611149,Eastbound McKnight Boulevard at 2 Street NW,-1,McKNIGHT BOULEVARD & CENTRE STREET,177.629168,False
1,2016-12-06 21:36:00+00:00-114.020547951.0476343,Eastbound Memorial Drive approaching Deerfoot ...,0,MEMORIAL DRIVE & DEERFOOT TRAIL SB RAMP,162.516880,False
2,2016-12-06 23:25:00+00:00-114.058178551.04447099,Macleod Trail at 9 Avenue SE,1,9 AVENUE & MACLEOD TRAIL,1.019846,True
3,2016-12-06 23:26:00+00:00-114.030871751.02839263,Ogden Road at Bonnybrook Road SE,-1,26 AVENUE/BLACKFOOT TR RAMP & OGDEN ROAD/PORTL...,228.739773,False
4,2016-12-07 00:05:00+00:00-114.119583551.06639113,Southbound University Drive at Crowchild Trail NW,-1,13 AVENUE & UNIVERSITY DRIVE,206.595403,False
5,2016-12-07 00:08:00+00:00-114.062440251.10264898,Northbound Centre Street approaching 56 Avenue NE,2,56 AVENUE/NORTHMOUNT DRIVE & CENTRE STREET,1.768927,True
6,2016-12-07 00:31:00+00:00-113.977418350.93172001,130 Avenue and Deerfoot Trail SE,-1,130 AVENUE & DEERFOOT TRAIL,22.957113,True
7,2016-12-07 01:03:00+00:00-113.958503751.07631642,26 Avenue and 52 Street NE,3,26 AVENUE & 52 STREET,8.767209,True
8,2016-12-07 02:36:00+00:00-113.976916951.05277711,Westbound Memorial Drive and 39 Street SE,4,MEMORIAL DRIVE & MARLBOROUGH WAY/39 STREET,16.567336,True
9,2016-12-07 05:14:00+00:00-114.06036751.04831937,Eastbound 5 Avenue at 1 Street SE,5,5 AVENUE & 1 STREET,2.793424,True


## Traffic-Control Incident Summaries

Only incidents within the 75 m threshold are used when computing
traffic-control-level statistics.

In [21]:
# Keep only incidents accepted by the matching threshold.
matched_incidents = incidents[
    incidents["matched_to_signal"]
].copy()

control_stats = (
    matched_incidents
    .groupby("matched_signal_id")
    .agg(
        matched_incident_count=("incident_id", "size"),
        first_matched_incident=("start_dt_local", "min"),
        latest_matched_incident=("start_dt_local", "max"),
        active_years=("year", "nunique"),
        most_common_weekday=("weekday", first_mode),
        most_common_hour=("hour", first_mode),
        most_common_time_period=("time_period", first_mode),
        most_common_description=("description", first_mode),
        median_match_distance_m=(
            "nearest_signal_distance_m",
            "median",
        ),
    )
    .reset_index()
    .rename(
        columns={
            "matched_signal_id": "signal_id",
        }
    )
)

# Keep signal IDs integer-like after grouping nullable matched IDs.
control_stats["signal_id"] = (
    control_stats["signal_id"]
    .astype("Int64")
)

display(control_stats.head())

,signal_id,matched_incident_count,first_matched_incident,latest_matched_incident,active_years,most_common_weekday,most_common_hour,most_common_time_period,most_common_description,median_match_distance_m
0,1109000525,73,2016-12-10 11:53:00-07:00,2026-09-07 14:16:00-06:00,11,Monday,15,Evening Rush,Traffic incident.,68.930240
1,1109000526,10,2018-03-22 23:08:00-06:00,2025-09-25 16:07:00-06:00,7,Thursday,14,Evening Rush,Traffic incident.,28.415180
2,1109000527,3,2021-08-24 00:16:00-06:00,2024-03-15 15:04:00-06:00,3,Tuesday,0,Evening Rush,Traffic incident.,7.692864
3,1109000529,4,2020-07-28 14:47:00-06:00,2024-09-03 14:18:00-06:00,3,Tuesday,14,Midday,There is an incident involving a pedestrian- E...,8.165272
4,1109000539,2,2025-09-11 08:34:00-06:00,2025-12-29 06:52:00-07:00,1,Monday,6,Morning Rush,Traffic incident.,19.528516


In [22]:
# Merge statistics back onto all traffic controls.
traffic_control_summary = signals.merge(
    control_stats,
    on="signal_id",
    how="left",
)

# Controls with no matched incidents receive zero counts.
traffic_control_summary["matched_incident_count"] = (
    traffic_control_summary["matched_incident_count"]
    .fillna(0)
    .astype(int)
)

traffic_control_summary["active_years"] = (
    traffic_control_summary["active_years"]
    .fillna(0)
    .astype(int)
)

traffic_control_summary = (
    traffic_control_summary
    .sort_values(
        "matched_incident_count",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    traffic_control_summary[
        [
            "signal_id",
            "intersection_name",
            "control_type",
            "matched_incident_count",
            "active_years",
            "most_common_weekday",
            "most_common_hour",
            "most_common_time_period",
            "median_match_distance_m",
        ]
    ].head(20)
)

,signal_id,intersection_name,control_type,matched_incident_count,active_years,most_common_weekday,most_common_hour,most_common_time_period,median_match_distance_m
0,1109001450,MEMORIAL DRIVE & DEERFOOT TRAIL SB RAMP,Traffic signal,454,11,Friday,17.0,Midday,46.668866
1,1109001431,16 AVENUE WB & DEERFOOT TRAIL SB RAMP,Traffic signal,435,11,Friday,17.0,Evening Rush,30.229005
2,1109001467,16 AVENUE & 19 STREET,Traffic signal,220,11,Thursday,14.0,Evening Rush,45.683314
3,1109001147,GLENMORE TRAIL & MACLEOD TRAIL,Traffic signal,205,11,Friday,16.0,Evening Rush,41.800250
4,1109001451,MEMORIAL DRIVE & DEERFOOT TRAIL SB RAMP,Traffic signal,183,10,Wednesday,17.0,Evening Rush,33.807053
5,1109001505,BARLOW TRAIL & PEIGAN TRAIL,Traffic signal,183,10,Tuesday,7.0,Morning Rush,70.614123
6,1110254720,BOW TRAIL & SARCEE TRAIL,Pedestrian RRFB,147,11,Tuesday,17.0,Evening Rush,55.562086
7,1109000786,5 AVENUE & CROWCHILD TRAIL,Traffic signal,130,10,Friday,16.0,Evening Rush,29.966314
8,1109001085,GLENMORE TRAIL & 5 STREET,Traffic signal,128,10,Tuesday,19.0,Evening,33.820944
9,1109001022,GLENMORE TRAIL & ELBOW DRIVE,Traffic signal,126,11,Tuesday,12.0,Evening Rush,54.350254


In [23]:
# Plot the 20 controls with the most matched incidents.
top_controls = (
    traffic_control_summary
    .head(20)
    .copy()
)

fig = px.bar(
    top_controls.sort_values("matched_incident_count"),
    x="matched_incident_count",
    y="intersection_name",
    orientation="h",
    title="20 Traffic Controls with Highest Matched Incident Counts",
    labels={
        "matched_incident_count": "Matched recorded incidents",
        "intersection_name": "Traffic-control location",
    },
)

fig.show()

## Spatial-Matching Validation

The following checks confirm that accepted matches obey the selected
75 m threshold and that all source incidents remain in the derived dataset.

In [24]:
# Validate hotspot and nearest-control outputs.
assert len(incidents) > 0
assert len(incidents) == len(incident_coords_rad)

assert incidents["hotspot_id"].notna().all()
assert incidents["nearest_signal_distance_m"].notna().all()

assert (
    incidents.loc[
        incidents["matched_to_signal"],
        "nearest_signal_distance_m",
    ]
    <= MATCH_RADIUS_M
).all()

assert hotspot_summary["hotspot_id"].is_unique
assert traffic_control_summary["signal_id"].is_unique

print("Spatial validation passed.")

Spatial validation passed.


## Export Derived Spatial Data

The outputs from this notebook are reused by the weather, risk-index,
modelling, and Streamlit stages.

In [25]:
# Output file locations.
INCIDENTS_SPATIAL_FILE = (
    CLEAN_DIR / "incidents_hotspots_intersections.csv"
)

HOTSPOT_SUMMARY_FILE = (
    CLEAN_DIR / "hotspot_summary.csv"
)

CONTROL_SUMMARY_FILE = (
    CLEAN_DIR / "traffic_control_incident_summary.csv"
)

# Save derived datasets.
incidents.to_csv(
    INCIDENTS_SPATIAL_FILE,
    index=False,
)

hotspot_summary.to_csv(
    HOTSPOT_SUMMARY_FILE,
    index=False,
)

traffic_control_summary.to_csv(
    CONTROL_SUMMARY_FILE,
    index=False,
)

print("Saved:")
print(INCIDENTS_SPATIAL_FILE)
print(HOTSPOT_SUMMARY_FILE)
print(CONTROL_SUMMARY_FILE)

Saved:
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\data\cleaned\incidents_hotspots_intersections.csv
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\data\cleaned\hotspot_summary.csv
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\data\cleaned\traffic_control_incident_summary.csv


## Hotspot and Intersection Analysis Summary

This stage added the main spatial layer of the project.

### Completed

- validated incident and traffic-control coordinates
- applied geographic DBSCAN using Haversine distance
- compared nearby DBSCAN parameter settings
- assigned incidents to hotspots or noise
- created hotspot-level descriptive summaries
- calculated nearest traffic controls using a BallTree
- reviewed nearest-control distance sensitivity
- applied a 75 m maximum matching threshold
- created traffic-control-level incident summaries
- exported reusable spatial datasets

### Interpretation Limits

Hotspot size and matched incident count describe **historical recorded
incident concentration**.

They are not direct measures of crash probability or intersection danger.
Traffic volume, road exposure, weather exposure, and other factors have
not yet been incorporated.

The next stage adds historical weather and builds richer risk features.